# Optimize SCALP-lite Parameters

Download datasets first with notebook 00. This notebook defines parameters, runs PCA then GPLVM latent Bayesian optimization, and saves the best settings for downstream notebooks.


In [ ]:
from scalp_lite.notebook_utils import (
    dataset_config,
    ensure_bo_dependencies,
    make_compact_search_space,
    make_estimator,
    make_scalp_optimization_objective,
    optimization_search_space,
    run_latent_bayesopt,
    save_best_optimization_result,
)


ensure_bo_dependencies()


In [ ]:
selected_dataset = "zebrafish"
random_state = 0
embedding_epochs = 100
invalid_score = -1e9

dataset = dataset_config(selected_dataset)
raw_estimator = make_estimator(dataset, n_components=100, random_state=random_state)
raw_adata = raw_estimator.to_data(dataset["input_path"])
raw_adata


In [ ]:
fixed_preprocess_params = {"max_cells": 2000}

base_preprocess_params = {
    "n_top_genes": 2000,
}

base_estimator_params = {
    "n_components": 100,
}

base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}


In [ ]:
preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 3000]},
}

estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 150]},
}

graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 40]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 8]},
    "assignment_quantile": {"type": "float", "bounds": [0.05, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 30]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

search_space = optimization_search_space(
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
)
search_space


In [ ]:
objective = make_scalp_optimization_objective(
    dataset=dataset,
    raw_adata=raw_adata,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
    embedding_epochs=embedding_epochs,
    invalid_score=invalid_score,
)


In [ ]:
pca_result = run_latent_bayesopt(
    objective,
    search_space,
    n_initial=12,
    latent_dim=3,
    n_iterations=12,
    embedding_model="pca",
    acquisition="ei",
    batch_size=1,
    random_state=random_state,
    invalid_score=invalid_score,
)

pca_result["best_params"], pca_result["best_score"]


In [ ]:
pca_result["history"].sort_values("score", ascending=False).head(10)


In [ ]:
compact_radii = {
    "n_top_genes": 500,
    "n_components": 30,
    "n_neighbors": 8,
    "intra_fraction": 0.12,
    "n_inter_edges": 2,
    "assignment_quantile": 0.15,
    "hubness_k": 6,
}

compact_search_space = make_compact_search_space(
    search_space,
    pca_result["best_params"],
    compact_radii,
    fix_categoricals=True,
)
compact_search_space


In [ ]:
gplvm_result = run_latent_bayesopt(
    objective,
    compact_search_space,
    n_initial=12,
    latent_dim=3,
    n_iterations=12,
    embedding_model="gplvm",
    acquisition="ei",
    batch_size=1,
    random_state=random_state + 1,
    invalid_score=invalid_score,
)

gplvm_result["best_params"], gplvm_result["best_score"]


In [ ]:
gplvm_result["history"].sort_values("score", ascending=False).head(10)


In [ ]:
params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params = save_best_optimization_result(
    dataset_name=selected_dataset,
    optimization_results={"pca": pca_result, "gplvm": gplvm_result},
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
)

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params
